# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[ 0.52779259 -0.52021294  0.93275245  0.27018602 -0.35368837]
 [ 0.51011815  0.70764127 -0.72245382  0.4809334  -0.06144767]
 [ 0.68626855 -0.28395193 -0.24021974  0.5863586  -0.27472513]
 [-0.47270332  0.0291788   0.46720373  0.09545299  0.7319706 ]
 [-0.01264105 -0.57349908  0.04683035 -0.7284884   0.114381  ]
 [-0.36179268  0.11809081  0.94369624  0.69124152 -0.50602418]
 [-0.9504625   0.54797741 -0.21777088 -0.49821547  0.17482341]
 [ 0.42611563 -0.46954029  0.18983071  0.39988962 -0.16400701]
 [-0.81865513  0.60355821 -0.39569763 -0.81097596 -0.99657086]
 [-0.20586979 -0.5189677  -0.25793693  0.61545501  0.16956291]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a1', 'a2', 'a1', 'a1', 'a2', 'a2', 'a1', 'a2', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 1, 1, 0, 1, 0, 0, 1, 0, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:01<00:34,  1.05s/it]

SVI:   3%|▎         | 1/34 [00:01<00:34,  1.05s/it, loss=2277.6829]

SVI:   6%|▌         | 2/34 [00:01<00:33,  1.05s/it, loss=3043.4768]

SVI:   9%|▉         | 3/34 [00:01<00:32,  1.05s/it, loss=2506.9392]

...

SVI:  84%|████████▍ | 21/25 [00:00<00:03,  1.10it/s, loss=2195.4858]

SVI:  88%|████████▊ | 22/25 [00:00<00:02,  1.10it/s, loss=2448.6912]

SVI:  92%|█████████▏| 23/25 [00:00<00:01,  1.10it/s, loss=2247.8999]

SVI:  96%|█████████▌| 24/25 [00:00<00:00,  1.10it/s, loss=2145.5654]

SVI: 100%|██████████| 25/25 [00:00<00:00,  1.10it/s, loss=2375.8054]